## Bussiness understanding

Build a Resume screening system

## Data Collection

we have 2 JD's & 10 Resumes collected for this project

## Data understanding

### Import necessary libraries

In [71]:
import torch
import transformers
import sentence_transformers
import docx
import pypdf
import pandas as pd
import numpy as np
import sklearn
import warnings
warnings.filterwarnings("ignore")

from tkinter import Tk
from tkinter.filedialog import askopenfilename, askopenfilenames

from docx import Document
from pypdf import PdfReader
from pathlib import Path


### Upload the files

In [72]:
from tkinter.filedialog import askopenfilename, askopenfilenames

jd_file = askopenfilename(title="Select Job Description")
resume_files = askopenfilenames(title="Select Resumes")

print("JD:", jd_file)
print("Resumes:", len(resume_files))

JD: R:/Learning/Core_NLP/05_NER_Entity_extraction/Resume_Screening/JD/JD 2.txt
Resumes: 12


### Extract the text

In [73]:
def extract_text(file_path):

    ext = Path(file_path).suffix.lower()

    if ext == ".txt":
        with open(file_path, "r", encoding="utf-8-sig") as f:
            return f.read()

    elif ext == ".docx":
        doc = Document(file_path)

        paragraphs = [p.text for p in doc.paragraphs]

        tables = [
            cell.text
            for table in doc.tables
            for row in table.rows
            for cell in row.cells
        ]

        return "\n".join(paragraphs + tables)

    elif ext == ".pdf":
        reader = PdfReader(file_path)

        return "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

    else:
        raise ValueError(f"Unsupported file format: {ext}")

### Extract JD and all resumes

In [74]:
# Extract Job Description
jd_data = {
    "filename": Path(jd_file).name,
    "text": extract_text(jd_file)
}

# Extract all Resumes
resumes_data = []

for file in resume_files:

    resumes_data.append({
        "filename": Path(file).name,
        "text": extract_text(file)
    })

print("JD Extraction Completed")
print("JD Characters:", len(jd_data["text"]))

print("\nResume Extraction Completed")
print("Total Resumes:", len(resumes_data))

for resume in resumes_data:
    print(resume["filename"], ":", len(resume["text"]), "characters")

JD Extraction Completed
JD Characters: 4442

Resume Extraction Completed
Total Resumes: 12
ADITYA SHARMA.docx : 3472 characters
ANANYA IYER.docx : 3332 characters
ARJUN MEHTA.docx : 3443 characters
DIVYA KRISHNAN.docx : 3243 characters
KARTHIK MENON.docx : 3051 characters
KAVYA REDDY.docx : 3724 characters
NEHA VERMA.docx : 3345 characters
PRIYA NAIR.docx : 4086 characters
ROHAN KAPOOR.docx : 3329 characters
SAMEER KHAN.docx : 3065 characters
SNEHA PATIL.docx : 3067 characters
VIKRAM RAO.docx : 3211 characters


### Inspect extracted text

In [75]:
print("JOB DESCRIPTION")
print(jd_data["text"][:1500])
print("="*150)
print("\nFIRST RESUME")
print(resumes_data[0]["filename"])
print(resumes_data[0]["text"][:1500])

JOB DESCRIPTION
## Title: Data Analyst – Business Intelligence

**Location:** Pune, India (Hybrid)
**Experience Required:** 2–5 Years
**Employment Type:** Full-Time

### About the Role

We are looking for a highly motivated **Data Analyst – Business Intelligence** to join our Analytics team. In this role, you will transform raw business data into meaningful insights that drive strategic decision-making across multiple business functions. You will collaborate with cross-functional teams, including Product, Marketing, Sales, Finance, and Operations, to understand business requirements, build insightful dashboards, perform statistical analyses, and provide actionable recommendations.

The ideal candidate is passionate about data, has strong analytical thinking, and enjoys solving complex business problems through data visualization and reporting. You should be comfortable working with large datasets, creating automated reporting solutions, and communicating insights to both technical and 

### Light pre-processing 

In [76]:
import re

def clean_text(text):

    # Normalize line breaks
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove null characters
    text = text.replace("\x00", " ")

    # Remove excessive spaces
    text = re.sub(r"[ \t]+", " ", text)

    # Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

### Clean JD and resumes

In [77]:
# Clean Job Description
jd_data["cleaned_text"] = clean_text(jd_data["text"])

# Clean all resumes
for resume in resumes_data:
    resume["cleaned_text"] = clean_text(resume["text"])

print("JD Cleaning Completed")
print("JD Characters:", len(jd_data["cleaned_text"]))

print("\nResume Cleaning Completed")

for resume in resumes_data:
    print(resume["filename"], ":", len(resume["cleaned_text"]))

JD Cleaning Completed
JD Characters: 4440

Resume Cleaning Completed
ADITYA SHARMA.docx : 3466
ANANYA IYER.docx : 3328
ARJUN MEHTA.docx : 3439
DIVYA KRISHNAN.docx : 3239
KARTHIK MENON.docx : 3047
KAVYA REDDY.docx : 3720
NEHA VERMA.docx : 3341
PRIYA NAIR.docx : 4082
ROHAN KAPOOR.docx : 3323
SAMEER KHAN.docx : 3061
SNEHA PATIL.docx : 3061
VIKRAM RAO.docx : 3207


In [78]:
print("CLEANED JOB DESCRIPTION")
print(jd_data["cleaned_text"][:1500])

print("\nCLEANED FIRST RESUME")
print(resumes_data[0]["cleaned_text"][:1500])

CLEANED JOB DESCRIPTION
## Title: Data Analyst – Business Intelligence

**Location:** Pune, India (Hybrid)
**Experience Required:** 2–5 Years
**Employment Type:** Full-Time

### About the Role

We are looking for a highly motivated **Data Analyst – Business Intelligence** to join our Analytics team. In this role, you will transform raw business data into meaningful insights that drive strategic decision-making across multiple business functions. You will collaborate with cross-functional teams, including Product, Marketing, Sales, Finance, and Operations, to understand business requirements, build insightful dashboards, perform statistical analyses, and provide actionable recommendations.

The ideal candidate is passionate about data, has strong analytical thinking, and enjoys solving complex business problems through data visualization and reporting. You should be comfortable working with large datasets, creating automated reporting solutions, and communicating insights to both techni

### Define Section Keywords

In [79]:
SECTION_KEYWORDS = {
    "skills": ["skills", "technical skills", "key skills","core competencies", "technologies"],
    "experience": ["experience", "work experience","professional experience", "employment history"],
    "projects": ["projects", "personal projects","academic projects", "key projects"],
    "education": ["education", "academic qualification","educational background"],
    "certifications": ["certifications", "certificates","professional certifications"],
    "summary": ["summary", "professional summary","career objective", "profile"]
}

### Create section identification function

In [80]:
def identify_sections(text):

    lines = text.split("\n")
    sections = {"other": []}
    current_section = "other"

    for line in lines:

        heading = line.strip().lower()
        heading = re.sub(r"[:\-]+$", "", heading).strip()

        matched_section = None

        for section, keywords in SECTION_KEYWORDS.items():
            if heading in keywords:
                matched_section = section
                break

        if matched_section:
            current_section = matched_section
            sections.setdefault(current_section, [])
        else:
            sections.setdefault(current_section, []).append(line)

    return {
        section: "\n".join(content).strip()
        for section, content in sections.items()
        if "\n".join(content).strip()
    }

### Identify sections in JD and resumes

In [81]:
jd_data["sections"] = identify_sections(jd_data["cleaned_text"])

for resume in resumes_data:
    resume["sections"] = identify_sections(resume["cleaned_text"])

print("JD Sections:")
for section, content in jd_data["sections"].items():
    print(section, ":", len(content), "characters")

print("\nResume Sections:")

for resume in resumes_data:
    print("\n", resume["filename"])

    for section, content in resume["sections"].items():
        print(section, ":", len(content), "characters")

JD Sections:
other : 4440 characters

Resume Sections:

 ADITYA SHARMA.docx
other : 209 characters
summary : 518 characters
skills : 460 characters
experience : 913 characters
projects : 446 characters
education : 644 characters
certifications : 170 characters

 ANANYA IYER.docx
other : 215 characters
summary : 540 characters
skills : 425 characters
experience : 828 characters
projects : 441 characters
education : 646 characters
certifications : 127 characters

 ARJUN MEHTA.docx
other : 190 characters
summary : 519 characters
skills : 448 characters
experience : 966 characters
projects : 456 characters
education : 621 characters
certifications : 133 characters

 DIVYA KRISHNAN.docx
other : 217 characters
summary : 477 characters
skills : 426 characters
experience : 770 characters
projects : 463 characters
education : 651 characters
certifications : 129 characters

 KARTHIK MENON.docx
other : 198 characters
summary : 473 characters
skills : 402 characters
experience : 688 characters
pro

In [82]:
for section, content in resumes_data[0]["sections"].items():

    print(f"\n{'='*50}")
    print(section.upper())
    print('='*50)
    print(content[:1000])


OTHER
ADITYA SHARMA
Senior Data Engineer
Bengaluru, India | aditya.sharma@email.com | +91-90000-00009 | linkedin.com/in/adityasharma
 
--------------------------------------------------------------------------------

SUMMARY
--------------------------------------------------------------------------------
Senior Data Engineer with 7 years of experience designing and operating large-scale
batch and streaming pipelines on AWS. Deep Spark, Airflow, and Kafka expertise, with
ownership of data warehousing and pipeline reliability for analytics and ML teams.
Track record of cutting pipeline cost and runtime while improving data quality and
freshness at scale.
 
--------------------------------------------------------------------------------

SKILLS
--------------------------------------------------------------------------------
Big Data & Streaming: Apache Spark, Apache Kafka, Spark Structured Streaming,
Databricks
Orchestration: Apache Airflow
Cloud (AWS): S3, Redshift, EMR, Glue
Warehouse:

### Load the embedding tokenizer

In [83]:
from transformers import AutoTokenizer

MODEL_NAME = "BAAI/bge-large-en-v1.5"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")
print("Maximum Model Length:", tokenizer.model_max_length)

Tokenizer loaded successfully!
Maximum Model Length: 512


### Create the chunking function

In [84]:
def create_chunks(text, chunk_size=400, overlap=50):

    sentences = re.split(r'(?<=[.!?])\s+|\n+', text)

    chunks = []
    current_chunk = []

    for sentence in sentences:

        sentence = sentence.strip()

        if not sentence:
            continue

        sentence_tokens = tokenizer.encode(
            sentence,
            add_special_tokens=False
        )

        current_tokens = tokenizer.encode(
            " ".join(current_chunk),
            add_special_tokens=False
        )

        if len(current_tokens) + len(sentence_tokens) <= chunk_size:

            current_chunk.append(sentence)

        else:

            if current_chunk:
                chunks.append(" ".join(current_chunk))

            # Preserve overlap from previous chunk
            previous_tokens = current_tokens[-overlap:]

            current_chunk = [
                tokenizer.decode(
                    previous_tokens,
                    skip_special_tokens=True
                ),
                sentence
            ]

            # Handle sentences larger than the chunk size
            combined_tokens = tokenizer.encode(
                " ".join(current_chunk),
                add_special_tokens=False
            )

            if len(combined_tokens) > chunk_size:
                current_chunk = []
                for i in range(0, len(sentence_tokens), chunk_size - overlap):
                    chunk_tokens = sentence_tokens[i:i + chunk_size]
                    chunks.append(
                        tokenizer.decode(
                            chunk_tokens,
                            skip_special_tokens=True
                        )
                    )

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

### Chunk the JD

In [85]:
jd_chunks = []

for section, content in jd_data["sections"].items():

    chunks = create_chunks(content)

    for chunk in chunks:

        jd_chunks.append({
            "section": section,
            "text": chunk
        })

print("Total JD Chunks:", len(jd_chunks))

for i, chunk in enumerate(jd_chunks[:5]):
    print(f"\nChunk {i+1} | Section: {chunk['section']}")
    print(chunk["text"][:300])

Total JD Chunks: 3

Chunk 1 | Section: other
## Title: Data Analyst – Business Intelligence **Location:** Pune, India (Hybrid) **Experience Required:** 2–5 Years **Employment Type:** Full-Time ### About the Role We are looking for a highly motivated **Data Analyst – Business Intelligence** to join our Analytics team. In this role, you will tra

Chunk 2 | Section: other
validation and quality checks to ensure accuracy, consistency, and reliability of reports. * create executive - level presentations and visualizations that clearly communicate business insights and recommendations. * monitor key business metrics and proactively identify anomalies or performance issu

Chunk 3 | Section: other
with * * google analytics * *, * * looker * *, * * metabase * *, or similar bi platforms. * knowledge of etl processes and data warehousing concepts. * understanding of data modeling and dimensional modeling techniques. * Experience with cloud platforms such as AWS, Azure, or Google Cloud Platform. 


### Chunk all resumes

In [86]:
for resume in resumes_data:

    resume["chunks"] = []

    for section, content in resume["sections"].items():

        chunks = create_chunks(content)

        for chunk in chunks:

            resume["chunks"].append({
                "section": section,
                "text": chunk
            })

print("Resume Chunking Completed\n")

for resume in resumes_data:

    print(
        resume["filename"],
        "| Total Chunks:",
        len(resume["chunks"])
    )

Resume Chunking Completed

ADITYA SHARMA.docx | Total Chunks: 7
ANANYA IYER.docx | Total Chunks: 7
ARJUN MEHTA.docx | Total Chunks: 7
DIVYA KRISHNAN.docx | Total Chunks: 7
KARTHIK MENON.docx | Total Chunks: 7
KAVYA REDDY.docx | Total Chunks: 7
NEHA VERMA.docx | Total Chunks: 7
PRIYA NAIR.docx | Total Chunks: 9
ROHAN KAPOOR.docx | Total Chunks: 7
SAMEER KHAN.docx | Total Chunks: 7
SNEHA PATIL.docx | Total Chunks: 7
VIKRAM RAO.docx | Total Chunks: 8


### Validate chunk sizes

In [87]:
print("JD Chunk Validation")

for i, chunk in enumerate(jd_chunks):

    token_count = len(
        tokenizer.encode(
            chunk["text"],
            add_special_tokens=True
        )
    )

    print(f"Chunk {i+1}: {token_count} tokens")


print("="*145)

for resume in resumes_data:

    print("\n", resume["filename"])

    for i, chunk in enumerate(resume["chunks"]):

        token_count = len(
            tokenizer.encode(
                chunk["text"],
                add_special_tokens=True
            )
        )

        print(
            f"Chunk {i+1} | "
            f"Section: {chunk['section']} | "
            f"Tokens: {token_count}"
        )

JD Chunk Validation
Chunk 1: 398 tokens
Chunk 2: 388 tokens
Chunk 3: 185 tokens

 ADITYA SHARMA.docx
Chunk 1 | Section: other | Tokens: 123
Chunk 2 | Section: summary | Tokens: 232
Chunk 3 | Section: skills | Tokens: 247
Chunk 4 | Section: experience | Tokens: 351
Chunk 5 | Section: projects | Tokens: 218
Chunk 6 | Section: education | Tokens: 388
Chunk 7 | Section: certifications | Tokens: 98

 ANANYA IYER.docx
Chunk 1 | Section: other | Tokens: 128
Chunk 2 | Section: summary | Tokens: 229
Chunk 3 | Section: skills | Tokens: 229
Chunk 4 | Section: experience | Tokens: 301
Chunk 5 | Section: projects | Tokens: 219
Chunk 6 | Section: education | Tokens: 384
Chunk 7 | Section: certifications | Tokens: 92

 ARJUN MEHTA.docx
Chunk 1 | Section: other | Tokens: 120
Chunk 2 | Section: summary | Tokens: 225
Chunk 3 | Section: skills | Tokens: 232
Chunk 4 | Section: experience | Tokens: 322
Chunk 5 | Section: projects | Tokens: 224
Chunk 6 | Section: education | Tokens: 392
Chunk 7 | Section: c

### Load the embedding model

In [88]:
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-large-en-v1.5"

device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(MODEL_NAME,device=device)

print("Embedding Model Loaded Successfully!")
print("Device:", device)
print("Embedding Dimensions:", embedding_model.get_sentence_embedding_dimension())
print("Maximum Sequence Length:", embedding_model.max_seq_length)

Loading weights: 100%|██████████| 391/391 [00:01<00:00, 245.98it/s]


Embedding Model Loaded Successfully!
Device: cuda
Embedding Dimensions: 1024
Maximum Sequence Length: 512


### Generate JD embeddings

In [89]:
jd_texts = [chunk["text"] for chunk in jd_chunks]

jd_embeddings = embedding_model.encode(
    jd_texts,
    batch_size=2,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("JD Embeddings Shape:", jd_embeddings.shape)

Batches: 100%|██████████| 2/2 [00:00<00:00,  3.53it/s]

JD Embeddings Shape: (3, 1024)


### Generate resume embeddings

In [90]:
for resume in resumes_data:

    resume_texts = [
        chunk["text"] for chunk in resume["chunks"]
    ]

    resume["embeddings"] = embedding_model.encode(
        resume_texts,
        batch_size=2,
        normalize_embeddings=True,
        show_progress_bar=False,
        convert_to_numpy=True
    )

print("Resume Embedding Generation Completed!")

for resume in resumes_data:
    print(
        resume["filename"],
        "| Embedding Shape:",
        resume["embeddings"].shape
    )

Resume Embedding Generation Completed!
ADITYA SHARMA.docx | Embedding Shape: (7, 1024)
ANANYA IYER.docx | Embedding Shape: (7, 1024)
ARJUN MEHTA.docx | Embedding Shape: (7, 1024)
DIVYA KRISHNAN.docx | Embedding Shape: (7, 1024)
KARTHIK MENON.docx | Embedding Shape: (7, 1024)
KAVYA REDDY.docx | Embedding Shape: (7, 1024)
NEHA VERMA.docx | Embedding Shape: (7, 1024)
PRIYA NAIR.docx | Embedding Shape: (9, 1024)
ROHAN KAPOOR.docx | Embedding Shape: (7, 1024)
SAMEER KHAN.docx | Embedding Shape: (7, 1024)
SNEHA PATIL.docx | Embedding Shape: (7, 1024)
VIKRAM RAO.docx | Embedding Shape: (8, 1024)


### Validate embeddings

In [91]:
print("JD Embeddings:", jd_embeddings.shape)

print("\nResume Embeddings:")

for resume in resumes_data:
    print(
        resume["filename"],
        resume["embeddings"].shape
    )

print("\nFirst JD Vector:")
print(jd_embeddings[0][:10])

JD Embeddings: (3, 1024)

Resume Embeddings:
ADITYA SHARMA.docx (7, 1024)
ANANYA IYER.docx (7, 1024)
ARJUN MEHTA.docx (7, 1024)
DIVYA KRISHNAN.docx (7, 1024)
KARTHIK MENON.docx (7, 1024)
KAVYA REDDY.docx (7, 1024)
NEHA VERMA.docx (7, 1024)
PRIYA NAIR.docx (9, 1024)
ROHAN KAPOOR.docx (7, 1024)
SAMEER KHAN.docx (7, 1024)
SNEHA PATIL.docx (7, 1024)
VIKRAM RAO.docx (8, 1024)

First JD Vector:
[-0.0103034   0.00096862 -0.03067248 -0.03313662 -0.02158637 -0.00434933
  0.0396799   0.0027696   0.0185666  -0.01481263]


### Calculate similarity for every resume chunk

In [104]:

similarity_results = []

for resume in resumes_data:

    # Compare every JD chunk with every resume chunk
    similarity_matrix = np.dot(
        jd_embeddings,
        resume["embeddings"].T
    )

    # Select the best JD match for each resume chunk
    scores = similarity_matrix.max(axis=0)

    for i, score in enumerate(scores):

        similarity_results.append({
            "candidate": resume["filename"],
            "section": resume["chunks"][i]["section"],
            "chunk_id": i + 1,
            "similarity_score": float(score)
        })

similarity_df = pd.DataFrame(similarity_results)

print("Similarity Calculation Completed!")
print("Total Comparisons:", len(similarity_df))


Similarity Calculation Completed!
Total Comparisons: 87


### Display chunk-level similarity

In [105]:
similarity_df = similarity_df.sort_values(
    by="similarity_score",
    ascending=False
)

display(similarity_df)

,candidate,section,chunk_id,similarity_score
51,PRIYA NAIR.docx,skills,3,0.790068
17,ARJUN MEHTA.docx,experience,4,0.782349
16,ARJUN MEHTA.docx,skills,3,0.780884
15,ARJUN MEHTA.docx,summary,2,0.779234
52,PRIYA NAIR.docx,experience,4,0.765852
...,...,...,...,...
34,KARTHIK MENON.docx,certifications,7,0.510622
13,ANANYA IYER.docx,certifications,7,0.507255
27,DIVYA KRISHNAN.docx,certifications,7,0.500625
86,VIKRAM RAO.docx,certifications,8,0.500137


### Calculate section-level similarity

In [106]:
section_similarity = (
    similarity_df
    .groupby(["candidate", "section"])["similarity_score"]
    .agg(["mean", "max", "count"])
    .reset_index()
)

section_similarity.columns = [
    "candidate",
    "section",
    "mean_similarity",
    "max_similarity",
    "chunk_count"
]

display(
    section_similarity.sort_values(
        by=["candidate", "section"]
    )
)

,candidate,section,mean_similarity,max_similarity,chunk_count
0,ADITYA SHARMA.docx,certifications,0.645593,0.645593,1
1,ADITYA SHARMA.docx,education,0.633317,0.633317,1
2,ADITYA SHARMA.docx,experience,0.655857,0.655857,1
3,ADITYA SHARMA.docx,other,0.570123,0.570123,1
4,ADITYA SHARMA.docx,projects,0.678603,0.678603,1
...,...,...,...,...,...
79,VIKRAM RAO.docx,experience,0.623351,0.623351,1
80,VIKRAM RAO.docx,other,0.520496,0.520496,1
81,VIKRAM RAO.docx,projects,0.583397,0.583397,1
82,VIKRAM RAO.docx,skills,0.565690,0.565690,1


### Display the top matching chunks

In [107]:
top_chunks = (
    similarity_df
    .sort_values("similarity_score", ascending=False)
    .groupby("candidate")
    .head(3)
)

display(top_chunks)

,candidate,section,chunk_id,similarity_score
51,PRIYA NAIR.docx,skills,3,0.790068
17,ARJUN MEHTA.docx,experience,4,0.782349
16,ARJUN MEHTA.docx,skills,3,0.780884
15,ARJUN MEHTA.docx,summary,2,0.779234
52,PRIYA NAIR.docx,experience,4,0.765852
50,PRIYA NAIR.docx,summary,2,0.757757
73,SNEHA PATIL.docx,summary,2,0.716278
68,SAMEER KHAN.docx,experience,4,0.705706
75,SNEHA PATIL.docx,experience,4,0.702430
30,KARTHIK MENON.docx,skills,3,0.694597


### Calculate weighted scores

In [108]:
# Copy similarity results
scoring_df = similarity_df.copy()

# Group sections into scoring categories
def get_category(section):

    if section == "skills":
        return "skills"

    elif section == "experience":
        return "experience"

    elif section == "projects":
        return "projects"

    else:
        return "other"


scoring_df["category"] = scoring_df["section"].apply(get_category)

# Calculate average similarity for each candidate and category
category_scores = (
    scoring_df
    .groupby(["candidate", "category"])["similarity_score"]
    .mean()
    .unstack(fill_value=0)
)

# Apply weightages
weights = {
    "skills": 0.40,
    "experience": 0.35,
    "projects": 0.15,
    "other": 0.10
}

# Ensure all scoring categories exist
category_scores = category_scores.reindex(
    columns=weights.keys(),
    fill_value=0
)

# Calculate final weighted score
category_scores["final_score"] = sum(
    category_scores[category] * weight
    for category, weight in weights.items()
)

# Convert to percentage-style scale for readability
category_scores["final_score"] *= 100

print("Weighted Candidate Scoring Completed!")

Weighted Candidate Scoring Completed!


In [109]:
final_ranking = (
    category_scores
    .reset_index()
    .sort_values("final_score", ascending=False)
    .reset_index(drop=True)
)

final_ranking.insert(
    0,
    "rank",
    range(1, len(final_ranking) + 1)
)

final_ranking["final_score"] = final_ranking["final_score"].round(2)

display(final_ranking)

category,rank,candidate,skills,experience,projects,other,final_score
0,1,ARJUN MEHTA.docx,0.780884,0.782349,0.669759,0.692542,75.59
1,2,PRIYA NAIR.docx,0.790068,0.696134,0.682344,0.679949,73.00
2,3,SNEHA PATIL.docx,0.677443,0.702430,0.658905,0.651034,68.08
3,4,KARTHIK MENON.docx,0.694597,0.691769,0.664749,0.598308,67.95
4,5,ADITYA SHARMA.docx,0.653574,0.655857,0.678603,0.622317,65.50
5,6,ROHAN KAPOOR.docx,0.652190,0.670693,0.571588,0.586735,64.00
6,7,SAMEER KHAN.docx,0.599130,0.705706,0.590778,0.607328,63.60
7,8,DIVYA KRISHNAN.docx,0.646196,0.645420,0.593766,0.597133,63.32
8,9,NEHA VERMA.docx,0.623741,0.633686,0.591900,0.597337,61.98
9,10,KAVYA REDDY.docx,0.613940,0.639085,0.601112,0.577218,61.71


### Display only Top 5 candidates

In [110]:
top_5_candidates = final_ranking.head(5)

display(top_5_candidates)

category,rank,candidate,skills,experience,projects,other,final_score
0,1,ARJUN MEHTA.docx,0.780884,0.782349,0.669759,0.692542,75.59
1,2,PRIYA NAIR.docx,0.790068,0.696134,0.682344,0.679949,73.00
2,3,SNEHA PATIL.docx,0.677443,0.702430,0.658905,0.651034,68.08
3,4,KARTHIK MENON.docx,0.694597,0.691769,0.664749,0.598308,67.95
4,5,ADITYA SHARMA.docx,0.653574,0.655857,0.678603,0.622317,65.50


### Load the Resume NER model

In [49]:
from transformers import pipeline

NER_MODEL = "yashpwr/resume-ner-bert"

ner_pipeline = pipeline(
    "ner",
    model=NER_MODEL,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

print("Resume NER Model Loaded Successfully!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 274.90it/s]


Resume NER Model Loaded Successfully!


### NER + content extraction

In [64]:

import re
import pandas as pd

HEADER_LINES = 40

def header_text(text):
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    return "\n".join(lines[:HEADER_LINES])[:1500]


def extract_contact(text):
    email = re.findall(
        r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}", text
    )

    phone = re.findall(
        r"(?:\+?\d{1,3}[\s\-]?)?(?:\(?\d{2,5}\)?[\s\-]?)?\d{3,5}[\s\-]?\d{4,6}",
        text
    )

    phone = [
        p.strip() for p in phone
        if 10 <= len(re.sub(r"\D", "", p)) <= 13
    ]

    linkedin = re.findall(
        r"(?:https?://)?(?:www\.)?linkedin\.com/in/[A-Za-z0-9\-_%]+",
        text, flags=re.I
    )

    github = re.findall(
        r"(?:https?://)?(?:www\.)?github\.com/[A-Za-z0-9\-_]+",
        text, flags=re.I
    )

    return {
        "email": email[0] if email else "",
        "phone": phone[0] if phone else "",
        "linkedin": linkedin[0] if linkedin else "",
        "github": github[0] if github else ""
    }


def run_ner(text):
    try:
        entities = ner_pipeline(text)
    except Exception as e:
        print("NER failed:", e)
        return {}

    output = {}

    for entity in entities:
        if entity["score"] < 0.60:
            continue

        label = entity["entity_group"]
        value = entity["word"].replace("##", "").strip()

        if len(value) >= 2:
            output.setdefault(label, [])
            if value not in output[label]:
                output[label].append(value)

    return output


def pick_name(ner_output, header, filename):
    for label, values in ner_output.items():
        if "NAME" in label.upper() or label.upper() in ("PER", "PERSON"):
            if values:
                return values[0].title()

    for line in header.split("\n")[:5]:
        if (1 < len(line.split()) <= 4
                and "@" not in line
                and not re.search(r"\d{4}", line)
                and not re.search(r"https?://|linkedin|github", line, re.I)):
            return line.title()

    return filename.rsplit(".", 1)[0].replace("_", " ").title()


In [70]:

candidate_rows = []
all_ner = []

top_candidates = final_ranking.head(5)

for _, row in top_candidates.iterrows():

    filename = row["candidate"]

    resume = next(
        r for r in resumes_data
        if r["filename"] == filename
    )

    original_text = resume["text"]
    head = header_text(original_text)

    entities = run_ner(head)
    contact = extract_contact(original_text)

    all_ner.append(entities)

    candidate_rows.append({
        "rank": row["rank"],
        "name": pick_name(entities, head, filename),
        "filename": filename,
        **contact
    })

candidate_info_df = pd.DataFrame(candidate_rows)
candidate_info_df.head(5)


,rank,name,filename,email,phone,linkedin,github
0,1,Sneha Patil,SNEHA PATIL.docx,sneha.patil@email.com,+91-90000-00010,linkedin.com/in/snehapatil,
1,2,Aditya Sharma,ADITYA SHARMA.docx,aditya.sharma@email.com,+91-90000-00009,linkedin.com/in/adityasharma,
2,3,Divya Krishnan,DIVYA KRISHNAN.docx,divya.krishnan@email.com,+91-90000-00012,linkedin.com/in/divyakrishnan,
3,4,Karthik Menon,KARTHIK MENON.docx,karthik.menon@email.com,+91-90000-00011,linkedin.com/in/karthikmenon,
4,5,Kavya Reddy,KAVYA REDDY.docx,kavya.reddy@email.com,+91-90000-00003,linkedin.com/in/kavyareddy,
